Initial NGSolve Practice Problem  

Given $u(x,y)=1-3x^2+2x^3$  

we set the boundary to Dirichlet =1 on the left side
and leave the others Neumann =0 default.  

Then we use $a(\nabla{u},\nabla{\phi})=(f,\phi)$ to solve for $u$.  

We set the RHS using the function then solve for our $u$.  

<span style="color:red">Current Concern: Direct Solve and CG Solve visually match, but don't match original function  
Consider error calculations in addition to visual inspection.</span>


In [ ]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import ipywidgets as widgets  # might use to make plots near each other for visual comparison
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);


In [ ]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and default=Neumann on the other 3 sides
fes = H1(mesh, order=1, dirichlet="left")
fes.ndof

# Set function that will be used for boundary and RHS
g = 1 - 3 * x * x + 2 * x * x * x
Draw(g, mesh)

In [ ]:
# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(g, BND) # this sets the Dirichlet boundary component to match the initial function (which is constant there), might want to modify later
Draw(gfu) 

In [ ]:
# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# Bilinear form
a = BilinearForm(fes)
a += grad(u)*grad(phi)*dx # laplacian 
a.Assemble()

print(a.mat)

In [ ]:
# Right hand side
f = LinearForm(1 * phi * dx).Assemble()  # got this version from some example problem
print(f.vec)

In [ ]:
# Solve by using u=A^-1 (f-Au)
r = f.vec - a.mat * gfu.vec
gfu.vec.data += a.mat.Inverse(freedofs=fes.FreeDofs()) * r
Draw(gfu)

In [ ]:
# Solve the PDE using CG
gfu1 = GridFunction(fes)
gfu1.Set(g, BND)
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu1, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu1)